# GNN Model

In [ ]:
import json
from pathlib import Path
from typing import Dict, List, Optional, Tuple

## GATWithAttributions

In [ ]:
class GATWithAttributions(nn.Module):
    """Graph Attention Network with dual output heads for classification and attribution."""

    def __init__(
        self,
        in_channels: int = 1,
        hidden_channels: int = 128,
        out_channels: int = 1,
        num_heads: int = 4,
        num_layers: int = 2,
        dropout: float = 0.3,
    ):
        """
        Initialize GAT model.

        Args:
        in_channels: Input feature dimension (default: 1 for abundance)
        hidden_channels: Hidden dimension for GAT conv layers
        out_channels: Output dimension for graph-level predictions
        num_heads: Number of attention heads
        num_layers: Number of GAT layers (default: 2)
        dropout: Dropout rate
        """
        super().__init__()
        self.in_channels = in_channels
        self.hidden_channels = hidden_channels
        self.out_channels = out_channels
        self.num_heads = num_heads
        self.dropout_rate = dropout

        # GAT Layers
        self.gat_layers = nn.ModuleList()

        # First layer: in_channels -> hidden_channels
        self.gat_layers.append(
            GATConv(
                in_channels=in_channels,
                out_channels=hidden_channels,
                heads=num_heads,
                concat=True,
                dropout=dropout,
                add_self_loops=True,
            )
        )

        # Intermediate layers
        for _ in range(num_layers - 1):
            self.gat_layers.append(
                GATConv(
                    in_channels=hidden_channels * num_heads,
                    out_channels=hidden_channels,
                    heads=num_heads,
                    concat=True if _ < num_layers - 2 else False,
                    dropout=dropout,
                    add_self_loops=True,
                )
            )

        # Classification Head (graph-level)
        gat_output_dim = (
            hidden_channels * num_heads if num_layers > 1 else hidden_channels
        )
        self.classification_head = nn.Sequential(
            nn.Linear(hidden_channels, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, out_channels),
        )

        # Attribution Head (node-level)
        self.attribution_head = nn.Linear(hidden_channels, 1)

        # Dropout layer for regularization
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        edge_attr: Optional[torch.Tensor] = None,
        batch: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Forward pass through GAT model.

        Args:
        x: Node features (num_nodes, in_channels)
        edge_index: Edge indices (2, num_edges)
        edge_attr: Edge attributes (num_edges, edge_dim)
        batch: Batch assignment for nodes

        Returns:
        Tuple of (graph_logits, node_importances)
        - graph_logits: (batch_size, 1) classification logits
        - node_importances: (num_nodes, 1) per-node importance scores
        """
        # GAT forward passes
        for i, gat_layer in enumerate(self.gat_layers):
            x = gat_layer(x, edge_index, edge_attr=edge_attr)
            if i < len(self.gat_layers) - 1:
                x = F.relu(x)
                x = self.dropout(x)

        # Node-level attribution scores
        node_importances = self.attribution_head(x)  # (num_nodes, 1)

        # Graph-level pooling for classification
        if batch is None:
            # Single graph case
            graph_features = global_mean_pool(x, batch=None)  # (1, hidden_channels)
        else:
            # Multiple graphs in batch
            graph_features = global_mean_pool(x, batch)  # (batch_size, hidden_channels)

        # Classification head
        graph_logits = self.classification_head(graph_features)  # (batch_size, 1)

        return graph_logits, node_importances.squeeze(-1)

    def get_attention_weights(self) -> List[torch.Tensor]:
        """Get attention weight matrices from GAT layers."""
        attention_weights = []
        for layer in self.gat_layers:
            if hasattr(layer, "att_src"):
                attention_weights.append(layer.att_src)
        return attention_weights


## ProteinGraphDataset

In [ ]:
class ProteinGraphDataset:
    """Dataset for loading protein graphs with per-sample features."""

    def __init__(
        self,
        graph_path: Path,
        processed_data_path: Path,
        seed: int = 42,
    ):
        """
        Initialize dataset.

        Args:
        graph_path: Path to GraphML graph file
        processed_data_path: Path to H5AD processed data
        seed: Random seed for reproducibility
        """
        self.graph_path = Path(graph_path)
        self.processed_data_path = Path(processed_data_path)
        self.seed = seed
        self.scaler = StandardScaler()

        # Load graph and data
        self._load_graph()
        self._load_data()
        self._prepare_samples()

    def _load_graph(self) -> None:
        """Load NetworkX graph from GraphML."""
        logger.info(f"Loading graph from {self.graph_path}")
        self.graph = nx.read_graphml(self.graph_path)
        self.protein_ids = list(self.graph.nodes())
        logger.info(f"Graph loaded: {len(self.protein_ids)} proteins, {self.graph.number_of_edges()} edges")

    def _load_data(self) -> None:
        """Load H5AD data with abundance matrix and labels."""
        logger.info(f"Loading processed data from {self.processed_data_path}")
        adata = anndata.read_h5ad(self.processed_data_path)

        # Extract abundance matrix (samples x proteins)
        self.abundance_data = pd.DataFrame(
            adata.X,
            columns=adata.var_names,
            index=adata.obs_names,
        )

        # Extract labels
        if "diagnosis" in adata.obs.columns:
            diagnosis = adata.obs["diagnosis"].values
            self.labels = np.array([1 if d == "AD" else 0 for d in diagnosis])
            self.sample_ids = adata.obs_names.tolist()
        else:
            raise ValueError("No diagnosis column in metadata")

        logger.info(f"Data loaded: {len(self.labels)} samples")
        logger.info(f"Class distribution: {np.bincount(self.labels)}")

    def _prepare_samples(self) -> None:
        """Prepare per-sample graph data."""
        # Filter abundance data to proteins in graph
        self.abundance_data = self.abundance_data[
            [p for p in self.abundance_data.columns if p in self.protein_ids]
        ]

        # Reorder to match graph protein order
        self.abundance_data = self.abundance_data[self.protein_ids]

        # Normalize features (independent for each sample)
        self.normalized_abundance = self.scaler.fit_transform(self.abundance_data)

        logger.info(
            f"Prepared {len(self.labels)} samples with "
            f"{len(self.protein_ids)} proteins in graph"
        )

    def get_graph_data_for_sample(self, sample_idx: int) -> Data:
        """
        Get PyG Data object for a single sample.

        Args:
        sample_idx: Sample index

        Returns:
        PyG Data object with node features, edges, and label
        """
        # Get node features for this sample
        node_features = torch.FloatTensor(
            self.normalized_abundance[sample_idx].reshape(-1, 1)
        )

        # Convert graph to PyG format
        pyg_graph = from_networkx(self.graph)

        # Ensure edge_attr is float tensor
        if pyg_graph.edge_attr is None:
            pyg_graph.edge_attr = torch.ones(
                pyg_graph.edge_index.shape[1], 1, dtype=torch.float
            )
        else:
            # Extract edge weights
            edge_weights = []
            for src, dst in pyg_graph.edge_index.T:
                src_id = self.protein_ids[src.item()]
                dst_id = self.protein_ids[dst.item()]
                weight = float(self.graph[src_id][dst_id].get("weight", 1.0))
                edge_weights.append(weight)
            pyg_graph.edge_attr = torch.FloatTensor(edge_weights).reshape(-1, 1)

        # Set node features and label
        pyg_graph.x = node_features
        pyg_graph.y = torch.tensor(self.labels[sample_idx], dtype=torch.float)

        return pyg_graph

    def get_train_val_test_split(
        self, test_size: float = 0.2, val_size: float = 0.2
    ) -> Tuple[List[int], List[int], List[int]]:
        """
        Get stratified train/val/test split indices.

        Args:
        test_size: Fraction for test set
        val_size: Fraction for validation set (from remaining after test)

        Returns:
        Tuple of (train_indices, val_indices, test_indices)
        """
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=self.seed)

        # First split: train+val vs test
        for train_val_idx, test_idx in skf.split(
            np.zeros(len(self.labels)), self.labels
        ):
            break

        # Second split: train vs val from train+val
        skf2 = StratifiedKFold(n_splits=5, shuffle=True, random_state=self.seed)
        for train_idx, val_idx in skf2.split(
            np.zeros(len(train_val_idx)),
            self.labels[train_val_idx]
        ):
            break

        return train_val_idx[train_idx].tolist(), train_val_idx[val_idx].tolist(), test_idx.tolist()

    def __len__(self) -> int:
        """Return number of samples."""
        return len(self.labels)

    def __getitem__(self, idx: int) -> Data:
        """Get a single sample."""
        return self.get_graph_data_for_sample(idx)


## GNNTrainer

In [ ]:
class GNNTrainer:
    """Orchestrates GNN training, validation, and evaluation."""

    def __init__(self, config_path: str = "config.yaml"):
        """
        Initialize trainer.

        Args:
        config_path: Path to configuration file
        """
        self.config = load_config(config_path)
        self.seed = self.config.get("debug", {}).get("seed", 42)

        # Training config (must be before _get_device)
        self.train_config = self.config.get("training", {})
        self.model_config = self.config.get("model", {})

        # Set random seeds
        np.random.seed(self.seed)
        torch.manual_seed(self.seed)

        # Device
        self.device = self._get_device()

        # Directories
        self.model_dir = Path(self.config["logging"]["checkpoint_dir"])
        self.model_dir.mkdir(parents=True, exist_ok=True)

        self.results_dir = Path(self.config["logging"]["checkpoint_dir"])
        self.results_dir.mkdir(parents=True, exist_ok=True)

    def _get_device(self) -> torch.device:
        """Get torch device (CUDA or CPU)."""
        device_name = self.train_config.get("device", "cuda")
        if device_name == "cuda" and not torch.cuda.is_available():
            logger.warning("CUDA not available, using CPU")
            return torch.device("cpu")
        return torch.device(device_name if device_name == "cpu" else "cuda")

    def train(self, cohorts: Optional[List[str]] = None) -> Dict:
        """
        Train GNN models on specified cohorts.

        Args:
        cohorts: List of cohort names

        Returns:
        Dictionary with training results
        """
        if cohorts is None:
            cohorts = ["ROSMAP", "MSBB", "MAYO"]

        all_results = {}

        for cohort in cohorts:
            logger.info(f"\n{'='*60}")
            logger.info(f"Training GNN for {cohort}")
            logger.info(f"{'='*60}")

            results = self._train_cohort(cohort)
            all_results[cohort] = results

        return all_results

    def _train_cohort(self, cohort: str) -> Dict:
        """Train GNN on a single cohort."""
        processed_dir = Path(self.config["data"]["processed_dir"])
        graphs_dir = Path("data/graphs")

        processed_file = processed_dir / f"{cohort}_processed.h5ad"
        graph_file = graphs_dir / f"{cohort}_graph.graphml"

        if not processed_file.exists() or not graph_file.exists():
            logger.warning(f"Missing data for {cohort}")
            return {}

        # Create dataset
        dataset = ProteinGraphDataset(
            graph_path=graph_file,
            processed_data_path=processed_file,
            seed=self.seed,
        )

        # Get split
        train_indices, val_indices, test_indices = dataset.get_train_val_test_split()

        logger.info(
            f"Split: train={len(train_indices)}, val={len(val_indices)}, test={len(test_indices)}"
        )

        # Create data loaders (batch_size=1 for per-sample training)
        train_graphs = [dataset[i] for i in train_indices]
        val_graphs = [dataset[i] for i in val_indices]
        test_graphs = [dataset[i] for i in test_indices]

        train_loader = DataLoader(train_graphs, batch_size=1, shuffle=True)
        val_loader = DataLoader(val_graphs, batch_size=1)
        test_loader = DataLoader(test_graphs, batch_size=1)

        # Initialize model
        model = GATWithAttributions(
            in_channels=1,
            hidden_channels=int(self.model_config.get("hidden_dims", [128])[0]),
            out_channels=1,
            num_heads=int(self.model_config.get("num_heads", 4)),
            num_layers=2,
            dropout=float(self.model_config.get("dropout_rate", 0.3)),
        ).to(self.device)

        logger.info(f"Model initialized: {model}")

        # Optimizer and scheduler
        optimizer = optim.Adam(
            model.parameters(),
            lr=float(self.train_config.get("learning_rate", 0.001)),
            weight_decay=float(
                self.train_config.get("weight_decay", 1e-5)
            ),
        )

        # Training loop
        num_epochs = int(self.train_config.get("epochs", 100))
        early_stopping_patience = int(self.train_config.get("early_stopping_patience", 20))

        best_val_loss = float("inf")
        patience_counter = 0
        training_history = {
            "loss": [],
            "val_loss": [],
            "train_auroc": [],
            "val_auroc": [],
        }

        for epoch in range(num_epochs):
            # Train
            train_loss = self._train_epoch(model, train_loader, optimizer)

            # Validate
            val_loss, val_metrics = self._evaluate(model, val_loader)

            # Log metrics
            training_history["loss"].append(train_loss)
            training_history["val_loss"].append(val_loss)
            training_history["train_auroc"].append(self._compute_auroc(model, train_loader))
            training_history["val_auroc"].append(val_metrics["auroc"])

            if epoch % 10 == 0:
                logger.info(
                    f"Epoch {epoch:03d}: Loss={train_loss:.4f}, "
                    f"Val Loss={val_loss:.4f}, Val AUROC={val_metrics['auroc']:.4f}"
                )

            # Early stopping
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                best_epoch = epoch

                # Save checkpoint
                torch.save(
                    {
                        "epoch": epoch,
                        "model_state": model.state_dict(),
                        "optimizer_state": optimizer.state_dict(),
                        "val_loss": val_loss,
                    },
                    self.model_dir / f"{cohort}_gnn_model.pt",
                )
            else:
                patience_counter += 1
                if patience_counter >= early_stopping_patience:
                    logger.info(f"Early stopping at epoch {epoch}")
                    break

        # Load best model and evaluate on test set
        checkpoint = torch.load(self.model_dir / f"{cohort}_gnn_model.pt")
        model.load_state_dict(checkpoint["model_state"])

        test_loss, test_metrics = self._evaluate(model, test_loader)

        logger.info(f"Test Metrics: {test_metrics}")

        # Save results
        results = {
            "best_epoch": best_epoch,
            "best_val_loss": float(best_val_loss),
            "test_metrics": test_metrics,
            "training_history": training_history,
        }

        self._save_results(cohort, results)

        return results

    def _train_epoch(self, model, data_loader, optimizer) -> float:
        """Train for one epoch."""
        model.train()
        total_loss = 0.0

        for batch in data_loader:
            batch = batch.to(self.device)
            optimizer.zero_grad()

            # Forward pass
            graph_logits, node_importances = model(
                batch.x, batch.edge_index, batch.edge_attr, batch.batch
            )

            # Reshape for loss computation
            # graph_logits: [batch_size, 1] or [1] depending on pooling
            # batch.y: [batch_size] or scalar
            logits_flat = graph_logits.view(-1)
            labels_flat = batch.y.view(-1)

            # Classification loss
            cls_loss = F.binary_cross_entropy_with_logits(
                logits_flat, labels_flat
            )

            # Auxiliary node importance loss (optional, reduces weight)
            node_loss = (
                0.1
                * F.mse_loss(
                    node_importances,
                    torch.zeros_like(node_importances),
                )
            )

            loss = cls_loss + node_loss
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        return total_loss / len(data_loader)

    def _evaluate(self, model, data_loader) -> Tuple[float, Dict]:
        """Evaluate model on a dataset."""
        model.eval()
        total_loss = 0.0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for batch in data_loader:
                batch = batch.to(self.device)

                graph_logits, _ = model(
                    batch.x, batch.edge_index, batch.edge_attr, batch.batch
                )

                # Reshape for loss computation
                logits_flat = graph_logits.view(-1)
                labels_flat = batch.y.view(-1)

                loss = F.binary_cross_entropy_with_logits(
                    logits_flat, labels_flat
                )
                total_loss += loss.item()

                # Predictions
                preds = torch.sigmoid(logits_flat).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(labels_flat.cpu().numpy())

        all_preds = np.array(all_preds)
        all_labels = np.array(all_labels)

        # Compute metrics
        try:
            auroc = roc_auc_score(all_labels, all_preds)
            auprc = average_precision_score(all_labels, all_preds)
        except ValueError:
            auroc = 0.5
            auprc = 0.5

        accuracy = accuracy_score(all_labels, (all_preds > 0.5).astype(int))
        f1 = f1_score(all_labels, (all_preds > 0.5).astype(int))

        metrics = {
            "loss": total_loss / len(data_loader),
            "auroc": auroc,
            "auprc": auprc,
            "accuracy": accuracy,
            "f1": f1,
        }

        return metrics["loss"], metrics

    def _compute_auroc(self, model, data_loader) -> float:
        """Compute AUROC on a dataset."""
        _, metrics = self._evaluate(model, data_loader)
        return metrics["auroc"]

    def _save_results(self, cohort: str, results: Dict) -> None:
        """Save training results."""
        results_path = self.results_dir / f"{cohort}_gnn_training.json"

        # Convert numpy types
        def convert_to_serializable(obj):
            if isinstance(obj, np.ndarray):
                return obj.tolist()
            elif isinstance(obj, (np.floating, np.integer)):
                return float(obj) if isinstance(obj, np.floating) else int(obj)
            elif isinstance(obj, dict):
                return {k: convert_to_serializable(v) for k, v in obj.items()}
            elif isinstance(obj, list):
                return [convert_to_serializable(item) for item in obj]
            return obj

        serializable_results = convert_to_serializable(results)

        with open(results_path, "w") as f:
            json.dump(serializable_results, f, indent=2)

        logger.info(f"Saved results to {results_path}")


## explain

In [ ]:
def explain(
    sample_idx: int,
    model: GATWithAttributions,
    dataset: ProteinGraphDataset,
    method: str = "attention",
) -> pd.DataFrame:
    """
    Generate explanations for a sample via node attribution.

    Args:
    sample_idx: Sample index
    model: Trained GAT model
    dataset: ProteinGraphDataset instance
    method: Attribution method ("attention" or "gradients")

    Returns:
    DataFrame with protein IDs and attribution scores
    """
    model.eval()
    device = next(model.parameters()).device

    # Get sample graph data
    data = dataset[sample_idx].to(device)

    with torch.no_grad():
        # Forward pass to get features
        x = data.x
        for i, gat_layer in enumerate(model.gat_layers):
            x = gat_layer(x, data.edge_index, edge_attr=data.edge_attr)
            if i < len(model.gat_layers) - 1:
                x = F.relu(x)

        # Get node importances
        node_importances = model.attribution_head(x).squeeze()
        attr_scores = node_importances.detach().cpu().numpy()

        # Normalize to 0-1
        if attr_scores.std() > 1e-6:
            attr_scores = (attr_scores - attr_scores.min()) / (
                attr_scores.max() - attr_scores.min() + 1e-6
            )
        else:
            attr_scores = np.ones_like(attr_scores) / len(attr_scores)

        # Return as DataFrame
        importance_df = pd.DataFrame(
            {
                "protein": dataset.protein_ids,
                "importance": attr_scores,
            }
        ).sort_values("importance", ascending=False)

        return importance_df
